<a href="https://colab.research.google.com/github/Abdullah-Farooq292/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-Farooq292/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — "The Anatomy of Growing Content" (Finding #1). The paper reports growing pages average 3.2K words vs 2.3K for declining pages, and recommends expanding thin-but-visible pages. Methodology question: the growing/declining label comes from a single 30-day-vs-prior-30-day impression comparison, not from an experiment where pages were actually expanded and then re-measured. Does the validation design support the causal recommendation ("expanding pages helps them grow"), or only the narrower correlational one ("growing pages tend to already be longer")? The paper's own Myth #4 finding (word count is only "Nuanced," not confirmed) supports treating this gap carefully.

Finding 2 — "The Freshness Multiplier" (Finding #4). The paper flags its own 361+ bucket as unstable (283:1 ratio on just 1 declining page) but reports the 31-90d window (7.88:1) as "the strongest measured growth window." Methodology question: was that window chosen because it's the most defensible cutoff, or because it produced the best-looking ratio among several candidate windows tested? The paper doesn't disclose how many window boundaries were tried before settling on 31-90. This is the same kind of question I should ask of my own Week-6 honest-split result below.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running my Week-5 model. "Before" = naive random train/test split (pages from the same client can land in both train and test). "After" = client-grouped split (GroupShuffleSplit, same as Week 5). Both use identical features, model config, and metric.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, subprocess
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
import numpy as np

if not os.path.isdir("flyrank-ml-internship"):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/Abdullah-Farooq292/flyrank-ml-internship"], check=True)
os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df)} rows")

# impressions_90d dropped — see Section 3 leakage audit
features = ["content_age_days", "days_since_last_update",
            "avg_position", "ctr", "word_count"]

X = df[features].replace([float("inf"), float("-inf")], None).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# BEFORE: naive random split (client leakage possible)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
tree_r = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_r.fit(X_train_r, y_train_r)
scores_r = tree_r.predict_proba(X_test_r)[:, 1]

# AFTER: client-grouped split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
tree_g = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_g.fit(X_train_g, y_train_g)
scores_g = tree_g.predict_proba(X_test_g)[:, 1]

print("BEFORE (random split, client overlap possible):")
for k in (20, 50):
    print(f"  Precision@{k}: {precision_at_k(scores_r, y_test_r.values, k):.3f}")

print("\nAFTER (client-grouped split, honest):")
for k in (20, 50):
    print(f"  Precision@{k}: {precision_at_k(scores_g, y_test_g.values, k):.3f}")

overlap = set(df["client_id"].iloc[train_idx]) & set(df["client_id"].iloc[test_idx])
print(f"\nClient overlap in grouped split (should be 0): {len(overlap)}")

Loaded 30000 rows
BEFORE (random split, client overlap possible):
  Precision@20: 0.750
  Precision@50: 0.800

AFTER (client-grouped split, honest):
  Precision@20: 0.650
  Precision@50: 0.600

Client overlap in grouped split (should be 0): 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same hunt as Week 3, applied to the final feature set: content_age_days, days_since_last_update, impressions_90d, avg_position, ctr, word_count. Checking each for (a) whether it could only be known after the outcome, and (b) window overlap with the label.

is_declining_label comes from a 30-day-vs-prior-30-day impressions comparison, but impressions_90d spans the full 90 days — which includes that same recent 30-day window used to build the label. That's partial temporal overlap: not the feature literally equaling the label, but part of the label's signal is baked into a feature that's supposed to be a predictor

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Correlation check — including impressions_90d as an excluded candidate
check_features = features + ["impressions_90d"]
for f in check_features:
    corr = df[f].corr(df["is_declining_label"])
    flag = "  <- excluded from final feature set" if f == "impressions_90d" else ""
    print(f"{f}: corr with label = {corr:.3f}{flag}")

print()
print("impressions_90d spans 90 days, which INCLUDES the most recent 30d")
print("used to construct is_declining_label -> partial window overlap.")
print("Excluded from final feature set as a precaution (see markdown above).")

content_age_days: corr with label = -0.164
days_since_last_update: corr with label = 0.081
avg_position: corr with label = -0.029
ctr: corr with label = -0.062
word_count: corr with label = 0.090
impressions_90d: corr with label = -0.018  <- excluded from final feature set

impressions_90d spans 90 days, which INCLUDES the most recent 30d
used to construct is_declining_label -> partial window overlap.
Excluded from final feature set as a precaution (see markdown above).


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Boldest original claim (Week 5, Section 4): "This shows the model is catching a real pattern that my single hand-written rule structurally cannot see."

Rewritten in safe language: Observed on this held-out client test set: the model assigns high confidence (>0.7) to 2,896 pages the baseline rule scores as zero, and of those, 58.6% were measured as actually declining. This is directional evidence the model captures signal the single-threshold baseline misses — it is decision-support input for prioritization, not proof of a causal mechanism, and it hasn't been validated outside this portfolio or time window.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [done ] Every section above is filled — markdown thinking AND the code that backs it
- [done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [done ] No client names, URLs, or private queries anywhere
- [done ] My claims use careful words: observed, measured, directional, decision-support
- [done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.